# NB08 — Ablation Study (standalone, Q1)

**Self-contained.** Does not depend on any other notebook. Given the split CSVs in `data/splits/`,
it trains everything it needs and writes every ablation figure and table straight into
`outputs/paper_q1/{figures,tables}` — so notebook 12 does **not** need to be run for these assets.

What this produces, all reviewer-driven:

1. **Component ablation with a CE-only reference and 3-seed CIs.** Reference is **cross-entropy only**
   (no FGM). We then add each component one at a time — **+FGM**, +focal+CW, +sampler, +MSD, +RDrop,
   +EMA — plus a full stack, on **BanglishBERT**, **3 seeds each**. This directly measures FGM's own
   contribution and reports mean ± std so the small deltas can be read against seed noise.
2. **Fusion ablation** (uniform average vs. Nelder–Mead weighted) — reuses the per-seed validation/test
   logits already saved during (1), no extra training.
3. **Script-mask ablation** (BanglaBERT masked off Romanized rows vs. voting everywhere) — also reuses
   saved logits.
4. **Taxonomy ablation** (5-class vs 9-class) under the proposed CE+FGM recipe.

Budget note: on an A5000 the component ablation is the cost (48 runs). Fusion and script-mask
ablations are logit arithmetic on cached outputs and cost nothing extra.


In [ ]:
# ── Imports, device, paths ───────────────────────────────────────────────────
import os, json, time, math, random, warnings, itertools
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, accuracy_score, matthews_corrcoef, roc_auc_score
from scipy.optimize import minimize
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA:", torch.cuda.is_available(), "| device:", device)

# Paths — adjust ROOT if your tree differs. Everything is relative to the repo root.
ROOT      = os.environ.get("PROJECT_ROOT", "..")
SPLIT_DIR = f"{ROOT}/data/splits"
OUT       = f"{ROOT}/outputs/ablation"
PAPER     = f"{ROOT}/outputs/paper_q1"
FIG       = f"{PAPER}/figures"
TAB       = f"{PAPER}/tables"
for d in (OUT, FIG, TAB):
    os.makedirs(d, exist_ok=True)
print("splits:", SPLIT_DIR)
print("assets ->", FIG, "and", TAB)


In [ ]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
DEBUG = False; DEBUG_SAMPLES = 400        # set True for a 2-epoch smoke test

ABLATION_MODEL = "banglishbert"           # component ablation runs on BanglishBERT
MODEL_PATH = {
    "banglishbert": "csebuetnlp/banglishbert",
    "banglabert":   "csebuetnlp/banglabert",
    "muril":        "google/muril-base-cased",
    "xlmr":         "xlm-roberta-base",
}[ABLATION_MODEL]

SEEDS = [42, 123, 456]                     # 3 seeds -> mean ± std for every config

# Plotting palette (matches the paper figures)
COL = {"blue":"#2f6fb0","green":"#1a9c6e","orange":"#e8930c","purple":"#7d5ba6",
       "red":"#d1495b","grey":"#8a8f98","text":"#1e2838","muted":"#6a7280"}
plt.rcParams.update({"font.size":11,"axes.spines.top":False,"axes.spines.right":False,
                     "axes.edgecolor":"#444","figure.dpi":120})

# BASE recipe. All component toggles OFF -> this is the CE-ONLY reference.
BASE = {"max_length":128,"batch_size":32,"eval_batch_size":128,"grad_accum_steps":1,"num_workers":2,
        "epochs":6,"encoder_lr":2e-5,"head_lr":8e-5,"weight_decay":0.01,"warmup_ratio":0.10,
        "max_grad_norm":1.0,"label_smoothing":0.03,"focal_gamma":2.0,"class_weight_beta":0.999,
        "dropout":0.25,"head_hidden_dim":384,"patience":3,"use_fp16":torch.cuda.is_available(),
        "sampler_alpha":0.5,"fgm_epsilon":1.0,"rdrop_alpha":0.5,"ema_decay":0.999,
        # component toggles (the ablation flips these); ALL OFF == CE only
        "use_focal":False,"use_cw":False,"use_balanced_sampler":False,"n_msd":1,
        "use_rdrop":False,"use_fgm":False,"use_ema":False}
TEXT = "text_clean"
if DEBUG:
    BASE["epochs"] = 2
    print("⚠ DEBUG MODE")
print("reference recipe = CE only (all toggles off); component ablation adds one at a time")


In [ ]:
# ── Training machinery (self-contained; mirrors the main training pipeline) ──
class DS(Dataset):
    def __init__(self, df, tok, maxlen, enc, label):
        self.t = df.reset_index(drop=True)[TEXT].fillna("").astype(str).tolist()
        self.y = [int(enc.get(v, -1)) for v in df[label]]
        self.tok, self.ml = tok, maxlen
    def __len__(self): return len(self.t)
    def __getitem__(self, i):
        e = self.tok(self.t[i], max_length=self.ml, truncation=True, padding=False)
        it = {k: torch.tensor(v, dtype=torch.long) for k, v in e.items()}
        it["label"] = torch.tensor(self.y[i], dtype=torch.long)
        return it

class Collator:
    def __init__(self, tok): self.tok = tok
    def __call__(self, fs):
        lb = torch.stack([f["label"] for f in fs])
        ft = [{k: v for k, v in f.items() if k != "label"} for f in fs]
        b = self.tok.pad(ft, padding=True, return_tensors="pt"); b["label"] = lb; return b

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, ls=0.0):
        super().__init__(); self.g, self.w, self.ls = gamma, weight, ls
    def forward(self, lg, t):
        ce = F.cross_entropy(lg, t, weight=self.w, reduction="none", label_smoothing=self.ls)
        return (((1 - torch.exp(-ce)) ** self.g) * ce).mean()

def build_cw(s, enc, beta=0.999, mw=10.0):
    m = s.map(enc).dropna().astype(int); n = len(enc); c = m.value_counts().sort_index()
    w = np.ones(n, dtype=np.float32)
    for i in range(n):
        k = int(c.get(i, 0))
        if k > 0: w[i] = 1.0 / max((1.0 - beta ** k) / max(1.0 - beta, 1e-12), 1e-9)
    w = np.clip(w, w.min(), w.min() * mw); w = w / w.mean()
    return torch.tensor(w, dtype=torch.float32, device=device)

def make_sampler(df, enc, label, alpha=0.5):
    y = df[label].map(enc).fillna(-1).astype(int).values
    cc = np.bincount(y[y >= 0], minlength=len(enc)).astype(float); cc[cc == 0] = 1.0
    cw = (1.0 / cc) ** float(alpha); sw = np.where(y >= 0, cw[np.clip(y, 0, None)], 0.0)
    return WeightedRandomSampler(torch.as_tensor(sw, dtype=torch.double), len(sw), replacement=True)

class MSDHead(nn.Module):
    def __init__(self, h, n, dp=0.25, inner=384, nm=4):
        super().__init__(); inner = min(inner, h)
        self.pre = nn.Sequential(nn.Linear(h, inner), nn.GELU(), nn.LayerNorm(inner))
        self.drops = nn.ModuleList([nn.Dropout(dp) for _ in range(max(1, nm))])
        self.out = nn.Linear(inner, n)
    def forward(self, x):
        h = self.pre(x)
        if self.training and len(self.drops) > 1:
            return torch.stack([self.out(d(h)) for d in self.drops], 0).mean(0)
        return self.out(self.drops[0](h))

class Encoder(nn.Module):
    def __init__(self, name, n, dp=0.25, inner=384, nm=4):
        super().__init__(); self.encoder = AutoModel.from_pretrained(name)
        h = self.encoder.config.hidden_size
        self._tti = self.encoder.config.model_type.lower() not in ("roberta", "xlm-roberta")
        self.head = MSDHead(h, n, dp, inner, nm)
    def _pool(self, o, m):
        hs = o.last_hidden_state
        return 0.5 * hs[:, 0] + 0.5 * ((hs * m.unsqueeze(-1).float()).sum(1) / m.sum(1, keepdim=True).float().clamp(1))
    def forward(self, ids, mask, tti=None):
        kw = {"input_ids": ids, "attention_mask": mask}
        if self._tti and tti is not None: kw["token_type_ids"] = tti
        return self.head(self._pool(self.encoder(**kw), mask))

def uparams(model, e, h, wd):
    nd = ["bias", "LayerNorm.weight", "LayerNorm.bias"]; g = []
    for grp, lr in [(model.encoder, e), (model.head, h)]:
        d, nde = [], []
        for n, p in grp.named_parameters():
            if p.requires_grad: (nde if any(x in n for x in nd) else d).append(p)
        g += [{"params": d, "lr": lr, "weight_decay": wd}, {"params": nde, "lr": lr, "weight_decay": 0.0}]
    return g

class FGM:
    def __init__(self, m, eps=1.0, key="word_embeddings"): self.m, self.e, self.k, self.b = m, eps, key, {}
    def attack(self):
        for n, p in self.m.named_parameters():
            if p.requires_grad and self.k in n and p.grad is not None:
                self.b[n] = p.data.clone(); nm = torch.norm(p.grad)
                if nm != 0 and not torch.isnan(nm): p.data.add_(self.e * p.grad / nm)
    def restore(self):
        for n, p in self.m.named_parameters():
            if n in self.b: p.data = self.b[n]
        self.b = {}

class EMA:
    def __init__(self, m, d=0.999):
        self.d = d; self.s = {n: p.detach().clone() for n, p in m.named_parameters() if p.requires_grad}; self.b = {}
    def update(self, m):
        for n, p in m.named_parameters():
            if p.requires_grad: self.s[n].mul_(self.d).add_(p.detach(), alpha=1 - self.d)
    def apply_shadow(self, m):
        self.b = {n: p.detach().clone() for n, p in m.named_parameters() if p.requires_grad}
        for n, p in m.named_parameters():
            if p.requires_grad: p.data.copy_(self.s[n])
    def restore(self, m):
        for n, p in m.named_parameters():
            if p.requires_grad and n in self.b: p.data.copy_(self.b[n])
        self.b = {}

def rdrop_kl(lp, lq):
    pl, ql = F.log_softmax(lp, -1), F.log_softmax(lq, -1); p, q = pl.exp(), ql.exp()
    return 0.5 * ((p * (pl - ql)).sum(-1) + (q * (ql - pl)).sum(-1)).mean()

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

@torch.no_grad()
def get_logits(model, loader):
    model.eval(); o = []
    for b in loader:
        b = {k: v.to(device) for k, v in b.items()}
        o.append(model(b["input_ids"], b["attention_mask"], b.get("token_type_ids")).cpu())
    return torch.cat(o)

@torch.no_grad()
def evaluate(model, loader, nc):
    model.eval(); P, Y, PR = [], [], []
    for b in loader:
        b = {k: v.to(device) for k, v in b.items()}
        pr = F.softmax(model(b["input_ids"], b["attention_mask"], b.get("token_type_ids")), -1).cpu().numpy()
        y = b["label"].cpu().numpy(); m = y >= 0
        P.extend(pr[m].argmax(-1)); Y.extend(y[m]); PR.extend(pr[m])
    y, p, pr = np.array(Y), np.array(P), np.array(PR)
    rec = {"macro_f1": round(float(f1_score(y, p, average="macro", zero_division=0)), 4),
           "weighted_f1": round(float(f1_score(y, p, average="weighted", zero_division=0)), 4),
           "accuracy": round(float(accuracy_score(y, p)), 4),
           "mcc": round(float(matthews_corrcoef(y, p)), 4)}
    try: rec["auroc"] = round(float(roc_auc_score(y, pr, multi_class="ovr", average="macro", labels=list(range(nc)))), 4)
    except Exception: rec["auroc"] = float("nan")
    return rec

print("machinery ready")


In [ ]:
# ── One training run. Returns test metrics AND saves val/test logits for reuse ──
# The saved logits let the fusion and script-mask ablations run later with NO extra training.
def run_config(cfg, label_col, train_df, val_df, test_df, enc, nc, tag, seed, save_logits_dir=None):
    set_seed(seed)
    tok = AutoTokenizer.from_pretrained(MODEL_PATH); coll = Collator(tok)
    lkw = dict(collate_fn=coll, num_workers=cfg["num_workers"], pin_memory=torch.cuda.is_available())
    ds = DS(train_df, tok, cfg["max_length"], enc, label_col)
    if cfg["use_balanced_sampler"]:
        tl = DataLoader(ds, batch_size=cfg["batch_size"],
                        sampler=make_sampler(train_df, enc, label_col, cfg["sampler_alpha"]),
                        shuffle=False, drop_last=True, **lkw)
    else:
        tl = DataLoader(ds, batch_size=cfg["batch_size"], shuffle=True, drop_last=True, **lkw)
    vl = DataLoader(DS(val_df,  tok, cfg["max_length"], enc, label_col), batch_size=cfg["eval_batch_size"], shuffle=False, **lkw)
    te = DataLoader(DS(test_df, tok, cfg["max_length"], enc, label_col), batch_size=cfg["eval_batch_size"], shuffle=False, **lkw)

    model = Encoder(MODEL_PATH, nc, cfg["dropout"], cfg["head_hidden_dim"], cfg["n_msd"]).to(device)
    cw = build_cw(train_df[label_col], enc, cfg["class_weight_beta"]) if cfg["use_cw"] else None
    crit = FocalLoss(cfg["focal_gamma"], cw, cfg["label_smoothing"]) if cfg["use_focal"] \
           else (lambda lg, t: F.cross_entropy(lg, t, weight=cw, label_smoothing=cfg["label_smoothing"]))
    opt = torch.optim.AdamW(uparams(model, cfg["encoder_lr"], cfg["head_lr"], cfg["weight_decay"]))
    ns = math.ceil(len(tl) / cfg["grad_accum_steps"]) * cfg["epochs"]
    sch = get_linear_schedule_with_warmup(opt, int(ns * cfg["warmup_ratio"]), ns)
    scaler = torch.amp.GradScaler("cuda") if (cfg["use_fp16"] and device.type == "cuda") else None
    fgm = FGM(model, cfg["fgm_epsilon"]) if cfg["use_fgm"] else None
    ema = EMA(model, cfg["ema_decay"]) if cfg["use_ema"] else None

    best, noimp, t0 = -1.0, 0, time.time()
    tmp = f"{OUT}/_tmp_{tag}_{seed}.pt"
    for ep in range(cfg["epochs"]):
        model.train(); opt.zero_grad(set_to_none=True)
        for step, b in enumerate(tl, 1):
            b = {k: v.to(device) for k, v in b.items()}; y = b["label"]
            with torch.autocast(device_type=device.type, enabled=scaler is not None):
                l1 = model(b["input_ids"], b["attention_mask"], b.get("token_type_ids"))
                if cfg["use_rdrop"]:
                    l2 = model(b["input_ids"], b["attention_mask"], b.get("token_type_ids"))
                    loss = 0.5 * (crit(l1, y) + crit(l2, y)) + cfg["rdrop_alpha"] * rdrop_kl(l1, l2)
                else:
                    loss = crit(l1, y)
                loss = loss / cfg["grad_accum_steps"]
            (scaler.scale(loss) if scaler else loss).backward()
            if fgm is not None:
                fgm.attack()
                with torch.autocast(device_type=device.type, enabled=scaler is not None):
                    la = crit(model(b["input_ids"], b["attention_mask"], b.get("token_type_ids")), y) / cfg["grad_accum_steps"]
                (scaler.scale(la) if scaler else la).backward(); fgm.restore()
            if step % cfg["grad_accum_steps"] == 0 or step == len(tl):
                if scaler: scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["max_grad_norm"])
                (scaler.step(opt), scaler.update()) if scaler else opt.step()
                sch.step(); opt.zero_grad(set_to_none=True)
                if ema is not None: ema.update(model)
        vm_raw = evaluate(model, vl, nc)["macro_f1"]; vm, use_ema = vm_raw, False
        if ema is not None and ep >= 2:
            ema.apply_shadow(model); vm_ema = evaluate(model, vl, nc)["macro_f1"]
            if vm_ema >= vm_raw: vm, use_ema = vm_ema, True
            ema.restore(model)
        if vm > best:
            best, noimp = vm, 0
            if use_ema: ema.apply_shadow(model)
            torch.save(model.state_dict(), tmp)
            if use_ema: ema.restore(model)
        else:
            noimp += 1
        if noimp >= cfg["patience"]: break

    model.load_state_dict(torch.load(tmp, map_location=device, weights_only=True))
    m = evaluate(model, te, nc); m["config"] = tag; m["seed"] = seed
    m["minutes"] = round((time.time() - t0) / 60, 1)

    # Save logits so fusion/script-mask ablations need no retraining.
    if save_logits_dir is not None:
        os.makedirs(save_logits_dir, exist_ok=True)
        torch.save(get_logits(model, vl).float(), f"{save_logits_dir}/val_logits.pt")
        torch.save(get_logits(model, te).float(), f"{save_logits_dir}/test_logits.pt")

    if os.path.exists(tmp): os.remove(tmp)
    del model; torch.cuda.empty_cache()
    print(f"  {tag:14s} seed{seed}: macroF1={m['macro_f1']} wF1={m['weighted_f1']} "
          f"acc={m['accuracy']} mcc={m['mcc']} [{m['minutes']}min]")
    return m

print("runner ready")


In [ ]:
# ── A. COMPONENT ABLATION — reference = CE only; add one component at a time ──
# Reference is plain cross-entropy (NO FGM). +FGM is the first tested addition, so FGM's own
# contribution is measured directly. 3 seeds per config -> mean ± std.
train_df = pd.read_csv(f"{SPLIT_DIR}/random_train.csv")
val_df   = pd.read_csv(f"{SPLIT_DIR}/random_val.csv")
test_df  = pd.read_csv(f"{SPLIT_DIR}/random_test.csv")
if DEBUG:
    train_df = pd.concat([g.sample(min(len(g), DEBUG_SAMPLES // 5), random_state=42)
                          for _, g in train_df.groupby("label5")]).reset_index(drop=True)
    val_df  = val_df.sample(min(len(val_df), 300), random_state=42)
    test_df = test_df.sample(min(len(test_df), 300), random_state=42)

classes5 = sorted(train_df["label5"].unique()); enc5 = {c: i for i, c in enumerate(classes5)}
print("5 classes:", enc5)

# Each config is a delta over the CE-only BASE.
CONFIGS = [
    ("CE only",      {}),                                                   # reference
    ("+FGM",         {"use_fgm": True}),                                    # <-- proves FGM's value
    ("+focal+CW",    {"use_fgm": True, "use_focal": True, "use_cw": True}),
    ("+sampler",     {"use_fgm": True, "use_balanced_sampler": True}),
    ("+MSD",         {"use_fgm": True, "n_msd": 4}),
    ("+RDrop",       {"use_fgm": True, "use_rdrop": True}),
    ("+EMA",         {"use_fgm": True, "use_ema": True}),
    ("Full stack",   {"use_fgm": True, "use_focal": True, "use_cw": True,
                      "use_balanced_sampler": True, "n_msd": 4, "use_rdrop": True, "use_ema": True}),
]

print(f"\nA. COMPONENT ABLATION (5-class, {ABLATION_MODEL}) — reference = CE only, {len(SEEDS)} seeds/config")
rows = []
for name, delta in CONFIGS:
    for sd in SEEDS:
        cfg = {**BASE, **delta}
        # For +FGM (the eventual proposed recipe) also cache logits for the fusion ablation.
        logdir = f"{OUT}/logits/{name.replace(' ','_').replace('+','p')}_seed{sd}" if name == "+FGM" else None
        m = run_config(cfg, "label5", train_df, val_df, test_df, enc5, 5, name, sd, save_logits_dir=logdir)
        rows.append(m)

raw = pd.DataFrame(rows)
raw.to_csv(f"{OUT}/component_ablation_perseed.csv", index=False)

# Aggregate: mean ± std across seeds
agg = (raw.groupby("config", sort=False)
          .agg(macro_f1_mean=("macro_f1","mean"), macro_f1_std=("macro_f1","std"),
               weighted_f1_mean=("weighted_f1","mean"),
               accuracy_mean=("accuracy","mean"), mcc_mean=("mcc","mean"))
          .reset_index())
ref_mean = float(agg.loc[agg["config"]=="CE only","macro_f1_mean"].iloc[0])
agg["delta_vs_CE"] = (agg["macro_f1_mean"] - ref_mean).round(4)
for c in ["macro_f1_mean","macro_f1_std","weighted_f1_mean","accuracy_mean","mcc_mean"]:
    agg[c] = agg[c].round(4)
agg.to_csv(f"{OUT}/component_ablation.csv", index=False)
print("\n", agg.to_string(index=False))
print(f"\n✅ saved {OUT}/component_ablation.csv  (+ per-seed csv)")


In [ ]:
# ── B. Train the 4 backbones (1 seed) to enable fusion + script-mask ablations ──
# We need real 4-backbone logits on val/test to compare (i) uniform vs weighted fusion and
# (ii) script-mask vs no-mask. One seed each is enough: these ablations are about the fusion
# rule and the masking, not seed variance. Logits are cached so both ablations are free.
FUSION_SEED = 42
BACKBONES = {"banglishbert":"csebuetnlp/banglishbert","banglabert":"csebuetnlp/banglabert",
             "muril":"google/muril-base-cased","xlmr":"xlm-roberta-base"}

def train_backbone_for_fusion(mk, mname, train_df, val_df, test_df, enc, nc, seed):
    """Train one backbone with the proposed CE+FGM recipe; cache val+test logits + script masks."""
    set_seed(seed)
    global MODEL_PATH
    saved = MODEL_PATH; MODEL_PATH = mname   # run_config reads MODEL_PATH
    cfg = {**BASE, "use_fgm": True}
    logdir = f"{OUT}/fusion_logits/{mk}_seed{seed}"
    m = run_config(cfg, "label5", train_df, val_df, test_df, enc, nc, mk, seed, save_logits_dir=logdir)
    MODEL_PATH = saved
    return logdir

# reuse the same random split loaded in cell 5
enc = enc5; nc = 5
print(f"\nB. Training 4 backbones (seed {FUSION_SEED}) for fusion + script-mask ablations")
fusion_dirs = {}
for mk, mname in BACKBONES.items():
    d = train_backbone_for_fusion(mk, mname, train_df, val_df, test_df, enc, nc, FUSION_SEED)
    fusion_dirs[mk] = d

# cache the label vectors and per-row script flags aligned to the saved logits
val_y  = val_df["label5"].map(enc).values
test_y = test_df["label5"].map(enc).values
def script_is_bangla(df):
    if "script" not in df.columns:
        return np.ones(len(df), bool)
    return df["script"].astype(str).str.lower().eq("bangla").values
val_bn  = script_is_bangla(val_df)
test_bn = script_is_bangla(test_df)
np.savez(f"{OUT}/fusion_meta.npz", val_y=val_y, test_y=test_y, val_bn=val_bn, test_bn=test_bn)
print("cached fusion meta (labels + script flags)")


In [ ]:
# ── C. Fusion ablation (uniform vs weighted) + script-mask ablation — no training ──
meta = np.load(f"{OUT}/fusion_meta.npz")
val_y, test_y, val_bn, test_bn = meta["val_y"], meta["test_y"], meta["val_bn"], meta["test_bn"]

# load cached logits: dict[name] -> tensor [N, C]
val_lg, test_lg = {}, {}
for mk in BACKBONES:
    d = f"{OUT}/fusion_logits/{mk}_seed{FUSION_SEED}"
    val_lg[mk]  = torch.load(f"{d}/val_logits.pt",  map_location="cpu").float()
    test_lg[mk] = torch.load(f"{d}/test_logits.pt", map_location="cpu").float()
names = list(val_lg.keys())
C = val_lg[names[0]].shape[1]

def macro(y, logits):
    return f1_score(y, logits.argmax(-1).numpy(), average="macro", zero_division=0)

# activation masks: BanglaBERT is the specialist. Two regimes:
#   masked   -> banglabert votes only on Bangla rows (mask off Romanized)
#   nomask   -> every backbone votes on every row
def build_act(bn_flags, mask_specialist):
    act = {}
    for mk in names:
        if mk == "banglabert" and mask_specialist:
            act[mk] = torch.tensor(bn_flags).float()
        else:
            act[mk] = torch.ones(len(bn_flags)).float()
    return act

def masked_ensemble(logits, act, w):
    num = den = None
    for i, mk in enumerate(names):
        a = act[mk].unsqueeze(1)
        t = w[i] * logits[mk] * a
        d = w[i] * a
        num = t if num is None else num + t
        den = d if den is None else den + d
    return num / (den + 1e-12)

def fit_weights(logits, act, y, restarts=30):
    k = len(names); best, bw = 1.0, np.ones(k) / k
    for _ in range(restarts):
        r = minimize(lambda rw: -macro(y, masked_ensemble(logits, act, np.abs(rw) + 1e-9)),
                     np.random.dirichlet(np.ones(k)), method="Nelder-Mead",
                     options={"maxiter": 1500, "xatol": 1e-5})
        if r.fun < best: best, bw = r.fun, np.abs(r.x) + 1e-9
    return bw / bw.sum()

# ---- C1. FUSION ABLATION (masking ON for both, as in the proposed system) ----
act_val_m  = build_act(val_bn,  mask_specialist=True)
act_test_m = build_act(test_bn, mask_specialist=True)

# best single backbone on test (for reference)
best_single = max((macro(test_y, test_lg[mk]), mk) for mk in names)
# uniform average
w_uniform = np.ones(len(names)) / len(names)
mf_uniform = macro(test_y, masked_ensemble(test_lg, act_test_m, w_uniform))
# weighted (Nelder-Mead, fit on validation)
w_opt = fit_weights(val_lg, act_val_m, val_y)
mf_weighted = macro(test_y, masked_ensemble(test_lg, act_test_m, w_opt))

fusion_rows = [
    {"Fusion method": "Best single backbone", "Test Macro-F1": round(best_single[0], 4), "Note": best_single[1]},
    {"Fusion method": "Uniform average",       "Test Macro-F1": round(mf_uniform, 4),      "Note": "equal weights"},
    {"Fusion method": "Nelder–Mead weighted",  "Test Macro-F1": round(mf_weighted, 4),     "Note": "validation-optimised"},
]
fusion_df = pd.DataFrame(fusion_rows)
fusion_df.to_csv(f"{TAB}/table_fusion_ablation.csv", index=False)
print("C1. FUSION ABLATION")
print(fusion_df.to_string(index=False))
print(f"   weighted - uniform = {mf_weighted - mf_uniform:+.4f}\n")

# ---- C2. SCRIPT-MASK ABLATION (weighted fusion for both; report Romanized + overall) ----
rom_test = ~test_bn.astype(bool)
def macro_subset(y, logits, subset):
    if subset.sum() == 0: return float("nan")
    return f1_score(y[subset], logits.argmax(-1).numpy()[subset], average="macro", zero_division=0)

# masked (proposed): specialist off Romanized rows
w_mask = fit_weights(val_lg, build_act(val_bn, True), val_y)
ens_mask = masked_ensemble(test_lg, build_act(test_bn, True), w_mask)
# no-mask: specialist votes everywhere
w_nomask = fit_weights(val_lg, build_act(val_bn, False), val_y)
ens_nomask = masked_ensemble(test_lg, build_act(test_bn, False), w_nomask)

scriptmask_rows = [
    {"Setting": "No script mask (all vote)",
     "Romanized Macro-F1": round(macro_subset(test_y, ens_nomask, rom_test), 4),
     "Overall Macro-F1":   round(macro(test_y, ens_nomask), 4)},
    {"Setting": "Script-aware mask (proposed)",
     "Romanized Macro-F1": round(macro_subset(test_y, ens_mask, rom_test), 4),
     "Overall Macro-F1":   round(macro(test_y, ens_mask), 4)},
]
scriptmask_df = pd.DataFrame(scriptmask_rows)
scriptmask_df.to_csv(f"{TAB}/table_scriptmask_ablation.csv", index=False)
print("C2. SCRIPT-MASK ABLATION")
print(scriptmask_df.to_string(index=False))
print(f"\n✅ saved fusion + script-mask ablation tables to {TAB}")


In [ ]:
# ── D. TAXONOMY ABLATION: 5-class vs 9-class under the proposed CE+FGM recipe ──
import re
NINE_MAP = {"none":"none","not bully":"none","sexual":"sexual","gender":"gender","religious":"religious",
            "religion":"religious","threat":"threat","calltoviolence":"threat","abusive/violence":"abusive",
            "abusive":"abusive","violence":"abusive","personal offense":"personal","personal":"personal",
            "political":"political","troll":"abusive","slander":"other","spam":"other","origin":"other",
            "body shaming":"other","misc":"other"}
NINE_PRIO = ["threat","sexual","religious","gender","political","abusive","personal","other","none"]
def to9(raw):
    if not isinstance(raw, str) or not raw.strip(): return "none"
    parts = [p.strip().lower() for p in re.split(r"[,_]", raw.strip()) if p.strip()]; b = set()
    for p in parts:
        if p in NINE_MAP: b.add(NINE_MAP[p])
        else:
            for k, v in NINE_MAP.items():
                if k in p: b.add(v); break
    if not b: return "other"
    for c in NINE_PRIO:
        if c in b: return c
    return "none"

for d in (train_df, val_df, test_df):
    d["label9"] = d["label_type"].apply(to9)
    d.loc[d["label_binary"] == 0, "label9"] = "none"
classes9 = sorted(set(train_df["label9"]) | set(val_df["label9"]) | set(test_df["label9"]))
enc9 = {c: i for i, c in enumerate(classes9)}
print("9 classes:", enc9)

OURS = {**BASE, "use_fgm": True}   # proposed recipe
tax_rows = []
# one seed is enough for the taxonomy headline (large effect); use seed 42
tax_rows.append(run_config(dict(OURS), "label5", train_df, val_df, test_df, enc5, 5, "5-class", 42))
tax_rows.append(run_config(dict(OURS), "label9", train_df, val_df, test_df, enc9, len(classes9), "9-class", 42))
tax_df = pd.DataFrame(tax_rows)[["config","macro_f1","weighted_f1","accuracy","mcc","auroc"]]
tax_df.to_csv(f"{OUT}/taxonomy_ablation.csv", index=False)
print("\nD. TAXONOMY ABLATION")
print(tax_df.to_string(index=False))
print(f"✅ saved {OUT}/taxonomy_ablation.csv")


In [ ]:
# ── E. Write paper_q1 figures + LaTeX tables (so NB12 is not needed for these) ──
def savefig(name):
    for ext in ("png", "pdf"):
        plt.savefig(f"{FIG}/{name}.{ext}", bbox_inches="tight", dpi=200)
    plt.close(); print(f"  fig -> {name}.png/.pdf")

def latex_table(df, path, caption, label, colfmt=None):
    cols = list(df.columns)
    colfmt = colfmt or ("l" + "c" * (len(cols) - 1))
    lines = [r"\begin{table}[!htbp]", r"\centering", f"\\caption{{{caption}}}", f"\\label{{{label}}}",
             r"\small", f"\\begin{{tabular}}{{@{{}}{colfmt}@{{}}}}", r"\toprule",
             " & ".join(f"\\textbf{{{c}}}" for c in cols) + r" \\", r"\midrule"]
    for _, r in df.iterrows():
        lines.append(" & ".join(str(r[c]) for c in cols) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    open(path, "w").write("\n".join(lines))
    print(f"  tex -> {os.path.basename(path)}")

# reload aggregated component ablation
agg = pd.read_csv(f"{OUT}/component_ablation.csv")
raw = pd.read_csv(f"{OUT}/component_ablation_perseed.csv")
order = ["CE only","+FGM","+focal+CW","+sampler","+MSD","+RDrop","+EMA","Full stack"]
agg["config"] = pd.Categorical(agg["config"], order, ordered=True)
agg = agg.sort_values("config").reset_index(drop=True)

# ---- Figure: component ablation with error bars (Macro-F1 mean ± std) ----
fig, ax = plt.subplots(figsize=(8.4, 4.4))
x = np.arange(len(agg))
y = agg["macro_f1_mean"].values
e = agg["macro_f1_std"].fillna(0).values
ce_ref = y[0]
ax.axhline(ce_ref, ls="--", lw=1.2, color=COL["grey"], label="CE-only reference", zorder=1)
bars = ax.errorbar(x, y, yerr=e, fmt="o-", color=COL["blue"], ecolor=COL["text"],
                   elinewidth=1.2, capsize=4, markersize=7, lw=2, zorder=3, label="Macro-F1 (mean ± std)")
for xi, yi, ei in zip(x, y, e):
    ax.text(xi, yi + ei + 0.0015, f"{yi:.4f}", ha="center", va="bottom", fontsize=8.4, color=COL["text"])
ax.set_xticks(x); ax.set_xticklabels(agg["config"], rotation=25, ha="right")
ax.set_ylabel("Macro-F1"); ax.set_title("Component ablation", fontweight="bold", loc="left")
ax.text(0, 1.02, f"BanglishBERT · {len(SEEDS)} seeds/config · reference = cross-entropy only",
        transform=ax.transAxes, fontsize=9.5, color=COL["muted"])
ax.legend(loc="lower left", fontsize=9, frameon=True)
savefig("fig06_component_ablation")

# ---- Table: component ablation (mean ± std, delta vs CE) ----
comp_tab = pd.DataFrame({
    "Configuration": agg["config"].astype(str),
    "Macro-F1": [f"{m:.4f} $\\pm$ {s:.4f}" if not np.isnan(s) else f"{m:.4f}"
                 for m, s in zip(agg["macro_f1_mean"], agg["macro_f1_std"])],
    "Weighted-F1": agg["weighted_f1_mean"].map(lambda v: f"{v:.4f}"),
    "Accuracy": agg["accuracy_mean"].map(lambda v: f"{v:.4f}"),
    "MCC": agg["mcc_mean"].map(lambda v: f"{v:.4f}"),
    "$\\Delta$ vs CE": agg["delta_vs_CE"].map(lambda v: f"{v:+.4f}"),
})
comp_tab.to_csv(f"{TAB}/table3_component_ablation.csv", index=False)
latex_table(comp_tab, f"{TAB}/table3_component_ablation.tex",
            "Component ablation on the 20\\% test split, BanglishBERT, three seeds per configuration "
            "(mean $\\pm$ std). The reference is cross-entropy only; each row adds one component. "
            "$\\Delta$ is the change in mean Macro-F1 relative to the CE reference.",
            "tab:component")

# ---- Table: taxonomy ablation ----
tax = pd.read_csv(f"{OUT}/taxonomy_ablation.csv")
tax_tab = tax.rename(columns={"config":"Taxonomy","macro_f1":"Macro-F1","weighted_f1":"Weighted-F1",
                              "accuracy":"Accuracy","mcc":"MCC","auroc":"AUROC"})
tax_tab.to_csv(f"{TAB}/table4_taxonomy.csv", index=False)
latex_table(tax_tab, f"{TAB}/table4_taxonomy.tex",
            "Taxonomy ablation under the proposed CE+FGM recipe. The finer nine-class scheme crashes "
            "Macro-F1 while leaving Weighted-F1 nearly unchanged, exposing unlearnable rare classes.",
            "tab:taxonomy")

# ---- Figure: taxonomy ablation ----
fig, ax = plt.subplots(figsize=(5.4, 4.2))
labs = ["5-class\nheadline", "9-class\nablation"]
vals = [float(tax.iloc[0]["macro_f1"]), float(tax.iloc[1]["macro_f1"])]
ax.bar(labs, vals, color=[COL["blue"], COL["grey"]], width=0.6)
for i, v in enumerate(vals):
    ax.text(i, v + 0.005, f"{v:.3f}", ha="center", va="bottom", fontsize=11)
ax.set_ylabel("Macro-F1"); ax.set_ylim(0.5, max(vals) + 0.06)
ax.set_title("Taxonomy ablation", fontweight="bold", loc="left")
savefig("fig07_taxonomy_ablation")

# ---- Tables: fusion + script-mask ----
fusion_df = pd.read_csv(f"{TAB}/table_fusion_ablation.csv")
latex_table(fusion_df, f"{TAB}/table_fusion_ablation.tex",
            "Fusion ablation on the 20\\% test split. Validation-optimised weighted-logit fusion is "
            "compared against a uniform average and the best single backbone.",
            "tab:fusion")
scriptmask_df = pd.read_csv(f"{TAB}/table_scriptmask_ablation.csv")
latex_table(scriptmask_df, f"{TAB}/table_scriptmask_ablation.tex",
            "Script-mask ablation. Masking the Bangla-script specialist (BanglaBERT) off Romanized rows "
            "is compared against letting every backbone vote on every comment.",
            "tab:scriptmask")

print(f"\n✅ ALL ablation assets written to:\n   {FIG}\n   {TAB}")
print("   figures: fig06_component_ablation, fig07_taxonomy_ablation")
print("   tables : table3_component_ablation, table4_taxonomy, table_fusion_ablation, table_scriptmask_ablation")
